In [26]:
#source('/research_jude/rgs01_jude/groups/jxugrp/home/common/Lab_Members/WenhuoHu/script/utils.r')
base_dir = '/scratch/mjehangir/manuscript_figures/manuscript_data/'
setwd(base_dir)

In [28]:
# Load required libraries
library(tidyverse)
library(corrplot)
library(ggpubr)
library(tidyverse)
library(readxl)
library(DESeq2)
library(dplyr)
library(ggplot2)
library(gridExtra)
library(ggrepel)
library(ComplexHeatmap)
library(tidyr)
library(dplyr)
library(GGally)
library(data.table)  # For fread()

In [29]:
# Load data
# Adjust file paths as necessary
SV_data <- read.delim("SVs_greater_than_1bp_length.txt")
CNV_data <- read.delim("CNVs_TEL_merged_final_data.tsv", header = TRUE, sep = "\t")
aneuploidy_data <- read.delim("/scratch/mjehangir/Glioma_project/glioma_aneuploidy/Glioma_v2_CNV_stats.txt")
#aneuploidy_data <- aneuploidy_data[aneuploidy_data$arm_call %in% c(1, -1), ]

head(SV_data, n=2)
head(CNV_data, n=2)
head(aneuploidy_data, n=2)

,chr,start,end,size,type,filename,arm
,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>
1,chr1,54754,54755,38,INS,3988,p
2,chr1,66252,66253,247,INS,3988,p


,SampleID,chr,assigned_arm,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,2436A,chr1,chr1p,124048267,27820989,0,0,0.2242755,0.000000000,0,5221.000,1,p
2,2436A,chr1,chr1q,124339061,0,200151,0,0.0000000,0.001609719,0,5051.167,1,q


,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
1,6478A,chr4,4p,0,2,1.0508593,278.657
2,6298B,chr4,4p,0,2,0.9986684,163.379


In [30]:
# Assuming your data frame is named SV_data
SV_data$chr_arm <- paste(SV_data$chr, SV_data$arm, sep = "")

# Display the first few rows to verify the new column
head(SV_data, n=2)

,chr,start,end,size,type,filename,arm,chr_arm
,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
1,chr1,54754,54755,38,INS,3988,p,chr1p
2,chr1,66252,66253,247,INS,3988,p,chr1p


In [31]:
# Rename the 'filename' column to 'SampleID'
colnames(SV_data)[colnames(SV_data) == "filename"] <- "SampleID"

# Check the updated data frame
head(SV_data, n= 2)

,chr,start,end,size,type,SampleID,arm,chr_arm
,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
1,chr1,54754,54755,38,INS,3988,p,chr1p
2,chr1,66252,66253,247,INS,3988,p,chr1p


In [32]:
head(CNV_data, n=2)
head(aneuploidy_data, n=2)

,SampleID,chr,assigned_arm,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,2436A,chr1,chr1p,124048267,27820989,0,0,0.2242755,0.000000000,0,5221.000,1,p
2,2436A,chr1,chr1q,124339061,0,200151,0,0.0000000,0.001609719,0,5051.167,1,q


,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
1,6478A,chr4,4p,0,2,1.0508593,278.657
2,6298B,chr4,4p,0,2,0.9986684,163.379


In [33]:
chr_p_q_arm_size = read.csv(file = "/home/mjehangir/chm13_chrs_arms/chromosome_p_q_sorted_arms.tsv", header = TRUE, sep = "\t")

In [34]:
head(chr_p_q_arm_size,  n= 2)

,Chromosome,Start,End,Length,p_Arm,q_Arm
,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,chr1,0,124048267,124048267,p,NA
2,chr1,124048267,248387328,124339061,q,NA


In [35]:
colnames(chr_p_q_arm_size)[colnames(chr_p_q_arm_size) == "p_Arm"] <- "arm_size"
head(chr_p_q_arm_size,  n= 2)

,Chromosome,Start,End,Length,arm_size,q_Arm
,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,chr1,0,124048267,124048267,p,NA
2,chr1,124048267,248387328,124339061,q,NA


In [36]:
chr_p_q_arm_size = chr_p_q_arm_size %>% select(Chromosome, Length, arm_size) 

In [37]:
head(chr_p_q_arm_size,  n= 2)

,Chromosome,Length,arm_size
,<chr>,<int>,<chr>
1,chr1,124048267,p
2,chr1,124339061,q


In [38]:
names(chr_p_q_arm_size)


[1] "Chromosome" "Length"     "arm_size"

In [39]:
head(aneuploidy_data, n=2)

,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
1,6478A,chr4,4p,0,2,1.0508593,278.657
2,6298B,chr4,4p,0,2,0.9986684,163.379


In [40]:
colnames(aneuploidy_data)[colnames(aneuploidy_data) == "arm"] <- "chr_arm"

In [41]:
head(aneuploidy_data, n=2)

,SampleID,seqnames,chr_arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
1,6478A,chr4,4p,0,2,1.0508593,278.657
2,6298B,chr4,4p,0,2,0.9986684,163.379


In [80]:
# Assuming your data frame is named aneuploidy_data
aneuploidy_data$chr_arm <- paste0("chr", aneuploidy_data$chr_arm)

# Check the result
head(aneuploidy_data, n=2)

SampleID,seqnames,chr_arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
6478A,chr4,chrchr4p,0,2,1.0508593,278.657
6298B,chr4,chrchr4p,0,2,0.9986684,163.379


In [81]:
# Convert data.frames to data.tables if they aren't already
setDT(aneuploidy_data)
setDT(CNV_data)
setDT(SV_data)
setDT(chr_p_q_arm_size)

# (a) For aneuploidy_data, create a common key "chr_arm" by concatenating "seqnames" and "arm"
# For example, if seqnames is "chr1" and arm is "p", then chr_arm becomes "chr1p".
#aneuploidy_data[, chr_arm := paste0(seqnames, arm)]

# (b) For CNV_data, the key already exists in "assigned_arm" (e.g., "chr1p").
# No change is needed here.

# (c) For SV_data, the key already exists as "chr_arm".
# (Ensure that its format matches the others.)

# (d) For chr_p_q_arm_size, create a key by concatenating "Chromosome" and "arm_size"
# and rename the column "Length" to "arm_size_val".
chr_p_q_arm_size[, chr_arm := paste0(Length, arm_size)]
setnames(chr_p_q_arm_size, "Chromosome", "arm_size_val")


#########################
### 2. Summarize SV_data: Count SV Events per SampleID and Chromosome Arm
#########################

# Count the number of rows (SV events) per SampleID and chr_arm
sv_summary <- SV_data[, .(SV_count = .N), by = .(SampleID, chr_arm)]
# The .N operator counts the number of rows in each group


#########################
### 3. Merge the Datasets
#########################

# (a) Merge CNV_data with aneuploidy_data to bring in the SV-related call metrics.
# Join by SampleID and by matching CNV_data's "assigned_arm" to aneuploidy_data's "chr_arm".
merge1 <- merge(CNV_data,
                aneuploidy_data[, .(SampleID, chr_arm, arm_call, arm_num_seg, arm_cr_wmean)],
                by.x = c("SampleID", "assigned_arm"),
                by.y = c("SampleID", "chr_arm"),
                all.x = TRUE)

# (b) Merge merge1 with the SV summary.
# Here, join by SampleID and match CNV_data's "assigned_arm" to the SV summary's "chr_arm".
merge2 <- merge(merge1,
                sv_summary,
                by.x = c("SampleID", "assigned_arm"),
                by.y = c("SampleID", "chr_arm"),
                all.x = TRUE)

# (c) Merge merge2 with the arm size information from chr_p_q_arm_size.
# Again, join by matching CNV_data's "assigned_arm" (the key) with chr_p_q_arm_size's "chr_arm".
merged_df <- merge(merge2,
                   chr_p_q_arm_size[, .(chr_arm, arm_size_val)],
                   by.x = "assigned_arm",
                   by.y = "chr_arm",
                   all.x = TRUE)


#########################
### 4. Compute SV Density
#########################
merged_df[, SV_count := as.numeric(SV_count)]
merged_df[, arm_size_val := as.numeric(arm_size_val)]

# Replace any missing SV_count with 0 (if no SV event was recorded, .N will be NA)
merged_df[is.na(SV_count), SV_count := 0]

# Calculate SV_density as the number of SV events divided by the arm size.
merged_df[, SV_density := SV_count / arm_size_val]


#########################
### 5. Select the Desired Columns for Final Output
#########################

# You requested the following columns in the final output:
#   - SampleID, chr, arm, SV_density, proportion_gain, proportion_loss, final_average_TL_p75
#     (from CNV_data) and arm_call, arm_num_seg, arm_cr_wmean (from aneuploidy_data).
final_df <- merged_df[, .(SampleID, chr, arm, SV_density,
                          proportion_gain, proportion_loss, final_average_TL_p75,
                          arm_call, arm_num_seg, arm_cr_wmean)]

# View the final merged data table
head(final_df, n=2)


ERROR: Error in setnames(chr_p_q_arm_size, "Chromosome", "arm_size_val"): Items of 'old' not found in column names: [Chromosome]. Consider skip_absent=TRUE.


In [82]:
head(sv_summary)

SampleID,chr_arm,SV_count
<chr>,<chr>,<int>
3988,chr1p,1227
3988,chr1q,995
3988,chr2p,988
3988,chr2q,1229
3988,chr3p,740
3988,chr3q,853


In [44]:
head(merged_df)

assigned_arm,SampleID,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,SV_count,arm_size_val,SV_density
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
chr10p,3988,chr10,40649191,0,32237835,0,0.00000000,0.7930745,0,3769.75,10,p,1,4,1.2123989,631,NA,NA
chr10p,6266D,chr10,40649191,0,34039950,0,0.00000000,0.8374078,0,5948.00,10,p,1,8,1.3019724,321,NA,NA
chr10p,6269C,chr10,40649191,12414570,2803290,0,0.30540755,0.0689630,0,3778.50,10,p,0,3,1.0198296,250,NA,NA
chr10p,6285B,chr10,40649191,2202585,0,0,0.05418521,0.0000000,0,4350.00,10,p,0,5,1.0197489,261,NA,NA
chr10p,6298B,chr10,40649191,4004700,0,0,0.09851857,0.0000000,0,4781.50,10,p,0,3,0.9777848,309,NA,NA
chr10p,6314E,chr10,40649191,3203760,0,0,0.07881485,0.0000000,0,9573.00,10,p,0,NA,NA,208,NA,NA


In [45]:
unique(final_df$arm)

[1] "p" "q"

In [46]:
library(data.table)

# Assuming final_df is your data.table
fwrite(final_df, file = "final_merged_for_correlation_analysis_v1.tsv", sep = "\t")

In [47]:
head(final_df, n =2)

SampleID,chr,arm,SV_density,proportion_gain,proportion_loss,final_average_TL_p75,arm_call,arm_num_seg,arm_cr_wmean
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>
3988,chr10,p,NA,0.7930745,0,3769.75,1,4,1.212399
6266D,chr10,p,NA,0.8374078,0,5948.00,1,8,1.301972


In [48]:
# Load required libraries
library(data.table)      # if your final_df is a data.table
library(dplyr)           # for data manipulation
library(ComplexHeatmap)  # for plotting the heatmap
library(circlize)        # for the color mapping with colorRamp2

# Assume final_df is your data.table with columns: 
# SampleID, chr, arm, SV_density, proportion_gain, proportion_loss, 
# final_average_TL_p75, arm_call, arm_num_seg, arm_cr_wmean, etc.

# Optional: convert final_df to a tibble (if you prefer dplyr piping)
final_df <- as_tibble(final_df)


circlize version 0.4.16
CRAN page: https://cran.r-project.org/package=circlize
Github page: https://github.com/jokergoo/circlize
Documentation: https://jokergoo.github.io/circlize_book/book/

If you use it in published research, please cite:
Gu, Z. circlize implements and enhances circular visualization
  in R. Bioinformatics 2014.

This message can be suppressed by:
  suppressPackageStartupMessages(library(circlize))




In [49]:
head(final_df)

SampleID,chr,arm,SV_density,proportion_gain,proportion_loss,final_average_TL_p75,arm_call,arm_num_seg,arm_cr_wmean
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>
3988,chr10,p,NA,0.7930745,0,3769.75,1,4,1.2123989
6266D,chr10,p,NA,0.8374078,0,5948.00,1,8,1.3019724
6269C,chr10,p,NA,0.0689630,0,3778.50,0,3,1.0198296
6285B,chr10,p,NA,0.0000000,0,4350.00,0,5,1.0197489
6298B,chr10,p,NA,0.0000000,0,4781.50,0,3,0.9777848
6314E,chr10,p,NA,0.0000000,0,9573.00,0,NA,NA


In [50]:
# Ensure proper chromosome ordering: extract numeric part from the 'chr' column
final_df <- final_df %>%
  mutate(chrom_num = as.numeric(gsub("chr", "", chr)))

# Define the features of interest
#features <- c("proportion_gain", "proportion_loss", "arm_cr_wmean", "INS_density", "DEL_density", "DUP_density","INV_density")

# Revised helper function that checks for complete pairs before computing correlation
get_corr_matrix <- function(data, features, chroms = 1:22) {
  
  # Initialize an empty matrix to store correlations
  corr_matrix <- matrix(NA, nrow = length(features), ncol = length(chroms),
                        dimnames = list(features, paste0("chr", chroms)))
  
  for (ch in chroms) {
    sub_data <- data %>% filter(chrom_num == ch)
    
    # Optionally check if there are at least 3 rows overall
    if (nrow(sub_data) >= 3) {
      for (f in features) {
        # Identify complete cases for the two variables
        complete_idx <- complete.cases(sub_data[[f]], sub_data$final_average_TL_p75)
        # Only compute correlation if there are at least 2 complete pairs
        if (sum(complete_idx) >= 2) {
          r_val <- cor(sub_data[[f]][complete_idx], sub_data$final_average_TL_p75[complete_idx])
        } else {
          r_val <- NA
        }
        corr_matrix[f, paste0("chr", ch)] <- r_val
      }
    } else {
      # Not enough rows for this chromosome; correlations remain as NA
      for (f in features) {
        corr_matrix[f, paste0("chr", ch)] <- NA
      }
    }
  }
  
  return(corr_matrix)
}

In [52]:
head(final_df)


SampleID,chr,arm,SV_density,proportion_gain,proportion_loss,final_average_TL_p75,arm_call,arm_num_seg,arm_cr_wmean,chrom_num
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>
3988,chr10,p,NA,0.7930745,0,3769.75,1,4,1.2123989,10
6266D,chr10,p,NA,0.8374078,0,5948.00,1,8,1.3019724,10
6269C,chr10,p,NA,0.0689630,0,3778.50,0,3,1.0198296,10
6285B,chr10,p,NA,0.0000000,0,4350.00,0,5,1.0197489,10
6298B,chr10,p,NA,0.0000000,0,4781.50,0,3,0.9777848,10
6314E,chr10,p,NA,0.0000000,0,9573.00,0,NA,NA,10


In [53]:
#features <- c("SV_density", "proportion_gain", "proportion_loss", "arm_cr_wmean", "DEL_density", "DUP_density", "INV_density")


In [55]:
colnames(final_df)


[1] "SampleID"             "chr"                  "arm"                 
 [4] "SV_density"           "proportion_gain"      "proportion_loss"     
 [7] "final_average_TL_p75" "arm_call"             "arm_num_seg"         
[10] "arm_cr_wmean"         "chrom_num"

In [56]:
# Create separate datasets for p arm and q arm
p_arm <- final_df %>% filter(arm == "p")
q_arm <- final_df %>% filter(arm == "q")

# Compute the correlation matrices for each arm
corr_matrix_p <- get_corr_matrix(p_arm, features)
corr_matrix_q <- get_corr_matrix(q_arm, features)

ERROR: Error: object 'features' not found


In [57]:
head(corr_matrix_p)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'head': object 'corr_matrix_p' not found


In [58]:
# Define the color function for the heatmap: blue for negative, white for zero, red for positive correlations
col_fun <- colorRamp2(c(-1, 0, 1), c("blue", "white", "red"))


In [59]:

options(repr.plot.width = 12, repr.plot.height = 3, repr.plot.res = 200)

Heatmap(
  corr_matrix_p,
  name = "Correlation",
  col = col_fun,
  cluster_rows = FALSE,        # preserve feature order
  cluster_columns = FALSE,     # preserve chromosome order
  column_title = "p arm: Chromosome-wide Correlation",
  row_title = "Feature vs final_average_TL_p75",
  heatmap_legend_param = list(title = "r"),
  cell_fun = function(j, i, x, y, width, height, fill) {
    # Add the correlation value (formatted to 2 decimal places) in the center of the cell
    grid.text(sprintf("%.2f", corr_matrix_p[i, j]), x, y, gp = gpar(fontsize = 10))
    
    # Draw a black border around the cell
    grid.rect(x = x, y = y, width = width, height = height,
              gp = gpar(col = "black", fill = NA, lwd = 0.5))
  }
)


ERROR: Error: object 'corr_matrix_p' not found


In [60]:
head(corr_matrix_p, n=5)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'head': object 'corr_matrix_p' not found


In [61]:
options(repr.plot.width = 12, repr.plot.height = 3, repr.plot.res = 200)

Heatmap(
  corr_matrix_q,
  name = "Correlation",
  col = col_fun,
  cluster_rows = FALSE,        # preserve feature order
  cluster_columns = FALSE,     # preserve chromosome order
  column_title = "q arm: Chromosome-wide Correlation",
  row_title = "Feature vs final_average_TL_p75",
  heatmap_legend_param = list(title = "r"),
  cell_fun = function(j, i, x, y, width, height, fill) {
    # Add the correlation value formatted to 2 decimal places in the center of the cell
    grid.text(sprintf("%.2f", corr_matrix_q[i, j]), x, y, gp = gpar(fontsize = 10))
    
    # Draw a black border around the cell
    grid.rect(x = x, y = y, width = width, height = height,
              gp = gpar(col = "black", fill = NA, lwd = 0.5))
  }
)


ERROR: Error: object 'corr_matrix_q' not found


In [62]:
head(SV_data)

chr,start,end,size,type,SampleID,arm,chr_arm
<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
chr1,54754,54755,38,INS,3988,p,chr1p
chr1,66252,66253,247,INS,3988,p,chr1p
chr1,67911,68342,431,DEL,3988,p,chr1p
chr1,83975,83976,63,INS,3988,p,chr1p
chr1,136935,136936,244,INS,3988,p,chr1p
chr1,180608,180652,44,DEL,3988,p,chr1p


In [63]:
# Count number of SVs per SampleID, chr_arm, and type (INS, DEL, DUP, INV, BND)
sv_type_summary <- SV_data[, .(SV_type_count = .N), by = .(SampleID, chr_arm, type)]

# Reshape data to wide format where each SV type gets its own column
sv_type_wide <- dcast(sv_type_summary, SampleID + chr_arm ~ type, value.var = "SV_type_count", fill = 0)

In [65]:
head(sv_type_wide)
nrow(sv_type_wide)

SampleID,chr_arm,DEL,DUP,INS,INV
<chr>,<chr>,<int>,<int>,<int>,<int>
2436A,chr10p,320,0,0,0
2436A,chr10q,469,1,0,0
2436A,chr11p,288,0,0,0
2436A,chr11q,395,1,0,0
2436A,chr12p,207,0,0,2
2436A,chr12q,510,0,0,2


[1] 898

In [66]:
#########################
### 3. Merge Datasets
#########################

# (a) Merge CNV_data with aneuploidy_data to bring in the SV-related call metrics
merge1 <- merge(CNV_data,
                aneuploidy_data[, .(SampleID, chr_arm, arm_call, arm_num_seg, arm_cr_wmean)],
                by.x = c("SampleID", "assigned_arm"),
                by.y = c("SampleID", "chr_arm"),
                all.x = TRUE)

In [67]:
head(merge1)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0.00000000,0.1978753,2732.75,10,q,0,5,0.9870386
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0.00000000,0.0000000,6766.00,11,p,0,6,1.0067300
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0.00000000,0.0000000,5319.40,12,p,0,2,1.0909572
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0.00000000,0.0000000,6085.00,12,q,0,10,1.0029376
2436A,chr14q,chr14,89761231,1001600,0,0,0.01115849,0.00000000,0.0000000,3490.60,14,q,0,11,1.0114610
2436A,chr15q,chr15,82566565,0,1602464,0,0.00000000,0.01940815,0.0000000,4111.50,15,q,0,3,0.9820486


In [68]:
# (b) Merge with SV summary (all SVs count)
merge2 <- merge(merge1,
                sv_type_wide,
                by.x = c("SampleID", "assigned_arm"),
                by.y = c("SampleID", "chr_arm"),
                all.x = TRUE)

In [69]:
head(merge2)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,DEL,DUP,INS,INV
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0.00000000,0.1978753,2732.75,10,q,0,5,0.9870386,469,1,0,0
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0.00000000,0.0000000,6766.00,11,p,0,6,1.0067300,288,0,0,0
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0.00000000,0.0000000,5319.40,12,p,0,2,1.0909572,207,0,0,2
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0.00000000,0.0000000,6085.00,12,q,0,10,1.0029376,510,0,0,2
2436A,chr14q,chr14,89761231,1001600,0,0,0.01115849,0.00000000,0.0000000,3490.60,14,q,0,11,1.0114610,435,0,0,0
2436A,chr15q,chr15,82566565,0,1602464,0,0.00000000,0.01940815,0.0000000,4111.50,15,q,0,3,0.9820486,380,0,0,0


In [90]:
head(chr_p_q_arm_size)

arm_size_val,Length,arm_size,chr_arm
<chr>,<int>,<chr>,<chr>
chr1,124048267,p,124048267p
chr1,124339061,q,124339061q
chr10,40649191,p,40649191p
chr10,94108943,q,94108943q
chr11,52743313,p,52743313p
chr11,82384456,q,82384456q


In [91]:
head(merge2)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,DEL,DUP,INS,INV
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0.00000000,0.1978753,2732.75,10,q,0,5,0.9870386,469,1,0,0
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0.00000000,0.0000000,6766.00,11,p,0,6,1.0067300,288,0,0,0
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0.00000000,0.0000000,5319.40,12,p,0,2,1.0909572,207,0,0,2
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0.00000000,0.0000000,6085.00,12,q,0,10,1.0029376,510,0,0,2
2436A,chr14q,chr14,89761231,1001600,0,0,0.01115849,0.00000000,0.0000000,3490.60,14,q,0,11,1.0114610,435,0,0,0
2436A,chr15q,chr15,82566565,0,1602464,0,0.00000000,0.01940815,0.0000000,4111.50,15,q,0,3,0.9820486,380,0,0,0


In [92]:
# (c) Merge with chr_p_q_arm_size to include arm size
merged_df <- merge(merge2,
                   chr_p_q_arm_size[, .(chr_arm, arm_size_val)],
                   by.x = "assigned_arm",
                   by.y = "chr_arm",
                   all.x = TRUE)

In [93]:
head(merged_df)

chr,SampleID,assigned_arm,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,⋯,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,DEL,DUP,INS,INV,arm_size_val
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<int>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<chr>
chr1,2436A,chr1p,124048267,27820989,0,0,0.2242755,0.000000000,0.00000000,⋯,1,p,0,8,0.9899055,581,0,0,2,NA
chr1,2436A,chr1q,124339061,0,200151,0,0.0000000,0.001609719,0.00000000,⋯,1,q,0,4,1.0078589,551,1,0,0,NA
chr1,3188,chr1p,124048267,0,0,141506757,0.0000000,0.000000000,1.14073949,⋯,1,p,-1,5,0.7648934,330,3,0,23,NA
chr1,3188,chr1q,124339061,0,1000750,0,0.0000000,0.008048557,0.00000000,⋯,1,q,0,3,1.0635141,378,5,0,8,NA
chr1,3988,chr1p,124048267,53040015,0,1601208,0.4275756,0.000000000,0.01290794,⋯,1,p,0,9,0.9993812,523,0,701,3,NA
chr1,3988,chr1q,124339061,21416157,0,20615520,0.1722400,0.000000000,0.16580083,⋯,1,q,0,5,1.0037472,459,0,526,10,NA


In [94]:
head(merged_df)

chr,SampleID,assigned_arm,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,⋯,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,DEL,DUP,INS,INV,arm_size_val
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<int>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<chr>
chr1,2436A,chr1p,124048267,27820989,0,0,0.2242755,0.000000000,0.00000000,⋯,1,p,0,8,0.9899055,581,0,0,2,NA
chr1,2436A,chr1q,124339061,0,200151,0,0.0000000,0.001609719,0.00000000,⋯,1,q,0,4,1.0078589,551,1,0,0,NA
chr1,3188,chr1p,124048267,0,0,141506757,0.0000000,0.000000000,1.14073949,⋯,1,p,-1,5,0.7648934,330,3,0,23,NA
chr1,3188,chr1q,124339061,0,1000750,0,0.0000000,0.008048557,0.00000000,⋯,1,q,0,3,1.0635141,378,5,0,8,NA
chr1,3988,chr1p,124048267,53040015,0,1601208,0.4275756,0.000000000,0.01290794,⋯,1,p,0,9,0.9993812,523,0,701,3,NA
chr1,3988,chr1q,124339061,21416157,0,20615520,0.1722400,0.000000000,0.16580083,⋯,1,q,0,5,1.0037472,459,0,526,10,NA


In [76]:
head(SV_data)

chr,start,end,size,type,SampleID,arm,chr_arm
<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
chr1,54754,54755,38,INS,3988,p,chr1p
chr1,66252,66253,247,INS,3988,p,chr1p
chr1,67911,68342,431,DEL,3988,p,chr1p
chr1,83975,83976,63,INS,3988,p,chr1p
chr1,136935,136936,244,INS,3988,p,chr1p
chr1,180608,180652,44,DEL,3988,p,chr1p


In [78]:
#########################
### 4. Compute SV Density Per Type
#########################

# Replace NA in SV count columns with 0
sv_types <- c("INS", "DEL", "DUP", "INV", "BND")

for (sv in sv_types) {
  if (!sv %in% names(merged_df)) {
    merged_df[, (sv) := 0]  # If column is missing, create and fill with 0
  } else {
    merged_df[is.na(get(sv)), (sv) := 0]
  }
}

#merged_df[, SV_count := as.numeric(SV_count)]
merged_df[, arm_size_val := as.numeric(arm_size_val)]

# Compute density for each SV type (SV count / arm_size_val)
for (sv in sv_types) {
  merged_df[, paste0(sv, "_density") := get(sv) / arm_size_val]
    # merged_df[, paste0(sv, "_density") := get(sv) / final_average_TL_p75]
 
}

In [79]:
head(merged_df)

assigned_arm,SampleID,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,⋯,DUP,INS,INV,arm_size_val,BND,INS_density,DEL_density,DUP_density,INV_density,BND_density
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr10p,3988,chr10,40649191,0,32237835,0,0.00000000,0.7930745,0,⋯,0,314,2,NA,0,NA,NA,NA,NA,NA
chr10p,6266D,chr10,40649191,0,34039950,0,0.00000000,0.8374078,0,⋯,0,0,0,NA,0,NA,NA,NA,NA,NA
chr10p,6269C,chr10,40649191,12414570,2803290,0,0.30540755,0.0689630,0,⋯,3,0,22,NA,0,NA,NA,NA,NA,NA
chr10p,6285B,chr10,40649191,2202585,0,0,0.05418521,0.0000000,0,⋯,0,0,0,NA,0,NA,NA,NA,NA,NA
chr10p,6298B,chr10,40649191,4004700,0,0,0.09851857,0.0000000,0,⋯,0,0,1,NA,0,NA,NA,NA,NA,NA
chr10p,6314E,chr10,40649191,3203760,0,0,0.07881485,0.0000000,0,⋯,0,0,1,NA,0,NA,NA,NA,NA,NA


In [74]:
#########################
### 5. Select Final Output Columns
#########################

final_df <- merged_df[, .(SampleID, chr, arm, SV_density = (INS + DEL + DUP + INV + BND) / arm_size_val,
                          proportion_gain, proportion_loss, final_average_TL_p75,
                          arm_call, arm_num_seg, arm_cr_wmean,
                          INS_density, DEL_density, DUP_density, INV_density, BND_density, proportion_gain, proportion_loss)]

# View the final merged data table
head(final_df)
#write.table(final_df, "final_merged_for_correlation_analysis_v2-2.tsv", 
            #row.names = FALSE, sep = "\t", quote = FALSE)

ERROR: Error in (INS + DEL + DUP + INV + BND)/arm_size_val: non-numeric argument to binary operator


In [75]:
head(merged_df)

assigned_arm,SampleID,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,⋯,arm,arm_call,arm_num_seg,arm_cr_wmean,DEL,DUP,INS,INV,arm_size_val,BND
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<chr>,<dbl>
chr10p,3988,chr10,40649191,0,32237835,0,0.00000000,0.7930745,0,⋯,p,1,4,1.2123989,315,0,314,2,NA,0
chr10p,6266D,chr10,40649191,0,34039950,0,0.00000000,0.8374078,0,⋯,p,1,8,1.3019724,321,0,0,0,NA,0
chr10p,6269C,chr10,40649191,12414570,2803290,0,0.30540755,0.0689630,0,⋯,p,0,3,1.0198296,225,3,0,22,NA,0
chr10p,6285B,chr10,40649191,2202585,0,0,0.05418521,0.0000000,0,⋯,p,0,5,1.0197489,261,0,0,0,NA,0
chr10p,6298B,chr10,40649191,4004700,0,0,0.09851857,0.0000000,0,⋯,p,0,3,0.9777848,308,0,0,1,NA,0
chr10p,6314E,chr10,40649191,3203760,0,0,0.07881485,0.0000000,0,⋯,p,0,NA,NA,207,0,0,1,NA,0


In [71]:
head(merge1, n=2)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0,0.0000000,6766.00,11,p,0,6,1.0067300


In [72]:
head(SV_data)

,chr,start,end,size,type,filename,arm
,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>
1,chr1,54754,54755,38,INS,3988,p
2,chr1,66252,66253,247,INS,3988,p
3,chr1,67911,68342,431,DEL,3988,p
4,chr1,83975,83976,63,INS,3988,p
5,chr1,136935,136936,244,INS,3988,p
6,chr1,180608,180652,44,DEL,3988,p


In [74]:
head(merge1)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0.00000000,0.1978753,2732.75,10,q,0,5,0.9870386
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0.00000000,0.0000000,6766.00,11,p,0,6,1.0067300
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0.00000000,0.0000000,5319.40,12,p,0,2,1.0909572
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0.00000000,0.0000000,6085.00,12,q,0,10,1.0029376
2436A,chr14q,chr14,89761231,1001600,0,0,0.01115849,0.00000000,0.0000000,3490.60,14,q,0,11,1.0114610
2436A,chr15q,chr15,82566565,0,1602464,0,0.00000000,0.01940815,0.0000000,4111.50,15,q,0,3,0.9820486


In [63]:
# Load the data.table library
library(data.table)

# Step 1: Compute SV summary from SV_data
# Group by SampleID, chr_arm, and type, calculating the number of SVs (n_SV) and total size
SV_summary <- SV_data[, .(total_size = sum(size)), by = .(SampleID, chr_arm, type)]

# Step 2: Merge SV_summary with merge1
# Join on SampleID and assigned_arm (from merge1) = chr_arm (from SV_summary)
result <- merge(merge1, SV_summary, 
                by.x = c("SampleID", "assigned_arm"), 
                by.y = c("SampleID", "chr_arm"))

# Step 3: Calculate SV density
# SVs_density = (number of SVs * total size) / chromosome arm length
result[, SVs_density := (total_size) / Length]
#result[, SVs_density := total_size / (Length / 1e6)]


# Step 4: Select all columns from merge1 plus type and SVs_density, and rename type to SV_type
final_output <- result[, c(names(merge1), "type", "SVs_density"), with = FALSE]
setnames(final_output, "type", "SV_type")

# The final_output data.table now contains the desired result

Warning message in gsum(size):
“The sum of an integer column for a group was more than type 'integer' can hold so the result has been coerced to 'numeric' automatically for convenience.”


In [75]:
head(SV_summary)

SampleID,chr_arm,type,total_size
<chr>,<chr>,<chr>,<dbl>
3988,chr1p,INS,257607
3988,chr1p,DEL,256730
3988,chr1p,INV,1013050
3988,chr1q,DEL,194912
3988,chr1q,INS,176145
3988,chr1q,INV,65827602


In [76]:
head(result)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,type,total_size,SVs_density
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>,<chr>,<dbl>,<dbl>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386,DEL,159358,1693.3354
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386,DUP,23673,251.5489
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0,0.0000000,6766.00,11,p,0,6,1.0067300,DEL,104279,1977.1037
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0,0.0000000,5319.40,12,p,0,2,1.0909572,DEL,68002,1893.5909
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0,0.0000000,5319.40,12,p,0,2,1.0909572,INV,99738133,2777318.6171
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0,0.0000000,6085.00,12,q,0,10,1.0029376,DEL,164131,1684.9003


In [77]:
head(final_output)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,SV_type,SVs_density
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>,<chr>,<dbl>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386,DEL,1693.3354
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386,DUP,251.5489
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0,0.0000000,6766.00,11,p,0,6,1.0067300,DEL,1977.1037
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0,0.0000000,5319.40,12,p,0,2,1.0909572,DEL,1893.5909
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0,0.0000000,5319.40,12,p,0,2,1.0909572,INV,2777318.6171
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0,0.0000000,6085.00,12,q,0,10,1.0029376,DEL,1684.9003


In [78]:
#final_output %>% filter(SVs_density > 10)

In [79]:
head(result)

SampleID,assigned_arm,chr,Length,total_length_neutral,total_length_gain,total_length_loss,proportion_neutral,proportion_gain,proportion_loss,final_average_TL_p75,chr_order,arm,arm_call,arm_num_seg,arm_cr_wmean,type,total_size,SVs_density
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<dbl>,<chr>,<dbl>,<dbl>
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386,DEL,159358,1693.3354
2436A,chr10q,chr10,94108943,26230785,0,18621836,0.27872787,0,0.1978753,2732.75,10,q,0,5,0.9870386,DUP,23673,251.5489
2436A,chr11p,chr11,52743313,3002850,0,0,0.05693328,0,0.0000000,6766.00,11,p,0,6,1.0067300,DEL,104279,1977.1037
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0,0.0000000,5319.40,12,p,0,2,1.0909572,DEL,68002,1893.5909
2436A,chr12p,chr12,35911664,4003740,0,0,0.11148857,0,0.0000000,5319.40,12,p,0,2,1.0909572,INV,99738133,2777318.6171
2436A,chr12q,chr12,97412884,9809163,0,0,0.10069677,0,0.0000000,6085.00,12,q,0,10,1.0029376,DEL,164131,1684.9003


In [80]:
# Replace 'result' with the name of your dataframe
write.table(result, 
            file = "final_merged_for_correlation_analysis_v3.tsv", 
            sep = "\t", 
            row.names = FALSE, 
            col.names = TRUE, 
            quote = FALSE)